## Imports

In [1]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from torch_geometric.nn.inits import glorot, zeros
from dataset import NeuroGraphDataset
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score,
    confusion_matrix, balanced_accuracy_score,
    classification_report,
)
import math, random, copy
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

GLOBAL_SEED = 42
torch.manual_seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

/home/sysadm/Desktop/Sagnik&Silajeet/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Augmentation helpers

In [2]:
def add_feature_noise(x: Tensor, std: float = 0.02, training: bool = True) -> Tensor:
    if not training or std == 0.0:
        return x
    return x + torch.randn_like(x) * std


def feature_masking(x: Tensor, mask_rate: float = 0.15, training: bool = True) -> Tensor:
    if not training or mask_rate == 0.0:
        return x
    return x * torch.bernoulli(torch.full_like(x, 1.0 - mask_rate))


def oversample_minority(train_list, minority_label=1, repeat_factor=None):
    minority       = [d for d in train_list if int(d.y.item()) == minority_label]
    majority_count = sum(1 for d in train_list if int(d.y.item()) != minority_label)
    if repeat_factor is None:
        repeat_factor = max(1, round(majority_count / max(len(minority), 1)))
    augmented = list(train_list)
    for _ in range(repeat_factor - 1):
        augmented.extend(minority)
    n_min = sum(1 for d in augmented if int(d.y.item()) == minority_label)
    n_maj = len(augmented) - n_min
    print(f"  Oversampling: {len(minority)} pMCI×{repeat_factor} → "
          f"pMCI={n_min}  sMCI={n_maj}  total={len(augmented)}")
    return augmented

## 2. GNN Backbone

In [3]:
class EdgeAwareGATv2Conv(MessagePassing):
    def __init__(self, in_channels, out_channels, heads=4, edge_dim=1,
                 dropout=0.3, negative_slope=0.2, concat=True, bias=True):
        super().__init__(aggr="add", node_dim=0)
        self.in_channels = in_channels; self.out_channels = out_channels
        self.heads = heads; self.edge_dim = edge_dim
        self.dropout = dropout; self.negative_slope = negative_slope
        self.concat = concat
        self.lin_key   = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.lin_query = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.lin_edge  = nn.Linear(edge_dim,    heads * out_channels, bias=False)
        self.att = nn.Parameter(torch.empty(1, heads, 3 * out_channels))
        out_dim = heads * out_channels if concat else out_channels
        self.bias = nn.Parameter(torch.empty(out_dim)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        glorot(self.lin_key.weight); glorot(self.lin_query.weight)
        glorot(self.lin_edge.weight); glorot(self.att)
        if self.bias is not None: zeros(self.bias)

    def forward(self, x, edge_index, edge_attr):
        N, _ = x.size(); H, C = self.heads, self.out_channels
        x_key   = self.lin_key(x).view(N, H, C)
        x_query = self.lin_query(x).view(N, H, C)
        out = self.propagate(edge_index, x=(x_key, x_query),
                             edge_attr=edge_attr, size=(N, N))
        out = out.view(N, H, C)
        out = out.reshape(N, H * C) if self.concat else out.mean(dim=1)
        if self.bias is not None: out = out + self.bias
        return out

    def message(self, x_i, x_j, edge_attr, index, ptr, size_i):
        E, H, C = x_i.size(0), self.heads, self.out_channels
        e_feat  = self.lin_edge(edge_attr).view(E, H, C)
        alpha   = (torch.cat([x_i, x_j, e_feat], dim=-1) * self.att).sum(dim=-1)
        alpha   = F.leaky_relu(alpha, self.negative_slope)
        alpha   = softmax(alpha, index, num_nodes=size_i)
        alpha   = F.dropout(alpha, p=self.dropout, training=self.training)
        return (alpha.unsqueeze(-1) * x_j).view(E, H * C)


class GATv2Block(nn.Module):
    def __init__(self, in_ch, out_ch, heads=4, edge_dim=1, dropout=0.3):
        super().__init__()
        self.conv = EdgeAwareGATv2Conv(in_ch, out_ch, heads, edge_dim, dropout)
        D = heads * out_ch
        self.act  = nn.ELU()
        self.norm = nn.LayerNorm(D)
        self.res  = nn.Linear(in_ch, D, bias=False) if in_ch != D else nn.Identity()

    def forward(self, x, ei, ea):
        return self.norm(self.act(self.conv(x, ei, ea)) + self.res(x))


class GATv2Stream(nn.Module):
    def __init__(self, in_ch, hidden, heads=4, edge_dim=1, dropout=0.3):
        super().__init__()
        self.l1 = GATv2Block(in_ch,          hidden, heads, edge_dim, dropout)
        self.l2 = GATv2Block(heads * hidden, hidden, heads, edge_dim, dropout)

    def forward(self, x, ei, ea):
        h1 = self.l1(x, ei, ea)
        return h1, self.l2(h1, ei, ea)


class MidStageCrossAttention(nn.Module):
    """Bidirectional cross-attention. Preserved for LLM explainability payload."""
    def __init__(self, embed_dim, num_heads=4, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim; self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.W_Q_s      = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_K_m      = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_V_m      = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj_s = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_Q_m      = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_K_s      = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_V_s      = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj_m = nn.Linear(embed_dim, embed_dim, bias=False)
        self.norm_s    = nn.LayerNorm(embed_dim)
        self.norm_m    = nn.LayerNorm(embed_dim)
        self.attn_drop = nn.Dropout(dropout)

    def _mh(self, Q, K, V):
        N, H, Dh = Q.size(0), self.num_heads, self.head_dim
        Q = Q.view(N, H, Dh).transpose(0, 1)
        K = K.view(N, H, Dh).transpose(0, 1)
        V = V.view(N, H, Dh).transpose(0, 1)
        a = self.attn_drop(torch.softmax(
            torch.bmm(Q, K.transpose(1, 2)) * self.scale, dim=-1))
        return torch.bmm(a, V).transpose(0, 1).contiguous().view(N, self.embed_dim)

    def forward(self, hs, hm):
        Cs = self.out_proj_s(self._mh(self.W_Q_s(hs), self.W_K_m(hm), self.W_V_m(hm)))
        Cm = self.out_proj_m(self._mh(self.W_Q_m(hm), self.W_K_s(hs), self.W_V_s(hs)))
        return self.norm_s(hs + Cs), self.norm_m(hm + Cm)


class GlobalAttentionPooling(nn.Module):
    def __init__(self, embed_dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or max(embed_dim // 2, 8)
        self.gate_nn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 1))

    def forward(self, h):
        alpha = torch.softmax(self.gate_nn(h), dim=0)
        return (alpha * h).sum(dim=0), alpha.squeeze(-1)


class StreamAttentionFusion(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.gate = nn.Linear(2 * embed_dim, 1)

    def forward(self, hs, hm):
        a = torch.sigmoid(self.gate(torch.cat([hs, hm], -1))).squeeze(-1)
        return a * hs + (1.0 - a) * hm, a

## 3. BrainGATv2 — GNN backbone (returns h_fused + attention payload)

In [4]:
class InputProjector(nn.Module):
    def __init__(self, in_dim=512, proj_dim=64, dropout=0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, proj_dim), nn.LayerNorm(proj_dim),
            nn.ELU(), nn.Dropout(dropout))

    def forward(self, x): return self.proj(x)


class BrainGATv2(nn.Module):
    """
    CNN-GNN hybrid backbone.
    Returns h_fused (64-dim patient embedding) + full attention payload for LLM stage.
    Final classification is done by SVM in v5.
    """
    def __init__(self, in_channels=512, proj_dim=64, hidden_channels=16, heads=4,
                 edge_dim_spat=1, edge_dim_morph=1, dropout_proj=0.2,
                 dropout_gat=0.3, global_feat_dim=None, global_proj_dim=32):
        super().__init__()
        stream_dim = heads * hidden_channels  # 64

        self.projector    = InputProjector(in_channels, proj_dim, dropout_proj)
        self.spat_stream  = GATv2Stream(proj_dim, hidden_channels, heads,
                                         edge_dim_spat,  dropout_gat)
        self.morph_stream = GATv2Stream(proj_dim, hidden_channels, heads,
                                         edge_dim_morph, dropout_gat)
        self.cross_attn   = MidStageCrossAttention(stream_dim, heads, dropout_gat)
        self.pool_spat    = GlobalAttentionPooling(stream_dim)
        self.pool_morph   = GlobalAttentionPooling(stream_dim)
        self.fusion       = StreamAttentionFusion(stream_dim)

        # [MOD 1] LayerNorm immediately before the SVM extraction point.
        # Prevents the SVM from being overwhelmed by feature scale differences
        # caused by scanner/site variation that survived per-scan z-scoring.
        # This is the embedding-level equivalent of the BatchNorm1d fix.
        self.pre_svm_ln = nn.LayerNorm(stream_dim)

        # [MOD 2] x_global projection: fuse AD-pretrained whole-brain embedding.
        # global_feat_dim is set to the embedding dim (e.g. 256) if x_global
        # exists in the dataset, else None → falls back to h_fused only.
        self.global_proj = None
        if global_feat_dim is not None and global_feat_dim > 0:
            self.global_proj = nn.Sequential(
                nn.Linear(global_feat_dim, global_proj_dim),
                nn.LayerNorm(global_proj_dim), nn.ELU(),
            )
            embed_out_dim = stream_dim + global_proj_dim
        else:
            embed_out_dim = stream_dim

        # Lightweight auxiliary MLP used ONLY during GNN training with Focal Loss.
        # At inference time we discard this and use the SVM instead.
        self.aux_mlp = nn.Sequential(
            nn.Linear(embed_out_dim, embed_out_dim // 2), nn.ELU(),
            nn.Dropout(dropout_gat),
            nn.Linear(embed_out_dim // 2, 1))

    def forward(self, x, ei_spat, ea_spat, ei_morph, ea_morph,
                x_global=None, feat_noise=0.0, mask_rate=0.0):
        x_p = self.projector(x)
        x_p = feature_masking(x_p, mask_rate,  self.training)
        x_p = add_feature_noise(x_p, feat_noise, self.training)

        _, h2_s = self.spat_stream( x_p, ei_spat,  ea_spat)
        _, h2_m = self.morph_stream(x_p, ei_morph, ea_morph)

        h_sc, h_mc = self.cross_attn(h2_s, h2_m)

        h_ps, a_s = self.pool_spat( h_sc)
        h_pm, a_m = self.pool_morph(h_mc)
        h_fused, a_stream = self.fusion(h_ps, h_pm)

        # [MOD 1] Apply LayerNorm before extraction / concat
        h_normed = self.pre_svm_ln(h_fused)

        # [MOD 2] Fuse x_global if available
        if self.global_proj is not None and x_global is not None:
            g = self.global_proj(x_global)
            h_embed = torch.cat([h_normed, g], dim=-1)
        else:
            h_embed = h_normed

        logit = self.aux_mlp(h_embed).squeeze(-1)
        return {
            "logit":        logit,
            "h_fused":      h_embed,         # (64 or 96)-dim → SVM input
            "alpha_spat":   a_s,             # (N,)   → LLM explainability
            "alpha_morph":  a_m,             # (N,)   → LLM explainability
            "alpha_stream": a_stream,        # scalar → LLM explainability
        }

## 4. Focal Loss

In [5]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma

    def forward(self, logits, targets):
        p     = torch.sigmoid(logits)
        p_t   = p * targets + (1 - p) * (1 - targets)
        a_t   = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (-a_t * (1 - p_t) ** self.gamma * torch.log(p_t.clamp(1e-8))).mean()


def build_focal(labels, device):
    n_p = (labels == 1).sum().float()
    n_n = (labels == 0).sum().float()
    a   = (n_n / (n_p + n_n)).item()
    return FocalLoss(alpha=a, gamma=2.0).to(device)

## 5. GNN Training

In [6]:
N_EPOCHS   = 150   # more epochs — GNN needs time to learn with accum
ACCUM      = 8     # gradient accumulation steps (simulates batch_size=8)
FEAT_NOISE = 0.02
MASK_RATE  = 0.10  # reduced: was masking too aggressively, hurting signal


def train_gnn(model, train_list, criterion, optimizer, scheduler, device):
    """
    Train GNN for N_EPOCHS with gradient accumulation.
    Prints loss every 10 epochs so convergence is visible.
    """
    model.train()
    for epoch in range(1, N_EPOCHS + 1):
        items = list(train_list)
        random.shuffle(items)
        optimizer.zero_grad()
        ep_loss = 0.0

        for step, p in enumerate(items):
            x   = F.normalize(p.x.to(device), p=2, dim=1)
            es  = p.edge_index_spatial.to(device)
            as_ = p.edge_attr_spatial.view(-1, 1).to(device)
            em  = p.edge_index_morph.to(device)
            am  = p.edge_attr_morph.view(-1, 1).to(device)
            xg  = p.x_global.to(device) if hasattr(p, "x_global") else None
            lbl = p.y.to(device)

            out  = model(x, es, as_, em, am, x_global=xg,
                         feat_noise=FEAT_NOISE, mask_rate=MASK_RATE)
            loss = criterion(out["logit"].view(-1), lbl.view(-1))
            (loss / ACCUM).backward()
            ep_loss += loss.item()

            if (step + 1) % ACCUM == 0 or (step + 1) == len(items):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()

        scheduler.step()

        if epoch % 10 == 0 or epoch == 1:
            print(f"    Epoch {epoch:3d}/{N_EPOCHS}  "
                  f"loss={ep_loss/len(items):.4f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")


def build_optimizer(model, lr=2e-4, weight_decay=1e-4):
    """
    OneCycleLR: warms up to peak LR then cosine-decays to near zero.
    This is the correct schedule for fixed-epoch training — the LR
    never resets (unlike CosineAnnealingWarmRestarts which was causing
    the loss to plateau by resetting to peak LR at epoch 40).
    """
    opt = torch.optim.AdamW(model.parameters(), lr=lr,
                            weight_decay=weight_decay)
    # total_steps = N_EPOCHS (one scheduler step per epoch)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=lr,
        total_steps=N_EPOCHS,
        pct_start=0.15,        # 15% warmup
        anneal_strategy="cos",
        div_factor=10.0,       # start lr = max_lr / 10
        final_div_factor=100.0 # end lr   = max_lr / 1000
    )
    return opt, sched

## 6. Embedding extraction + SVM classifier

In [7]:
def extract_embeddings(model, data_list, device):
    """
    Extract h_fused embeddings and attention payloads for all patients.
    [MOD 2] x_global is now passed to forward() so the SVM sees the
    full fused representation (GNN local + AD-pretrained global).
    Returns (embeddings: np.ndarray [N, D], labels: np.ndarray [N],
             alpha_spat: list, alpha_morph: list, alpha_stream: list)
    """
    model.eval()
    embeddings, labels = [], []
    alpha_spat, alpha_morph, alpha_stream = [], [], []

    with torch.no_grad():
        for p in data_list:
            x  = F.normalize(p.x.to(device), p=2, dim=1)
            es = p.edge_index_spatial.to(device)
            as_= p.edge_attr_spatial.view(-1, 1).to(device)
            em = p.edge_index_morph.to(device)
            am = p.edge_attr_morph.view(-1, 1).to(device)
            # [MOD 2] Pass x_global if present in the Data object
            xg = p.x_global.to(device) if hasattr(p, "x_global") else None

            out = model(x, es, as_, em, am, x_global=xg,
                        feat_noise=0.0, mask_rate=0.0)
            embeddings.append(out["h_fused"].cpu().numpy())
            labels.append(int(p.y.item()))
            alpha_spat.append(out["alpha_spat"].cpu())
            alpha_morph.append(out["alpha_morph"].cpu())
            alpha_stream.append(out["alpha_stream"].item())

    return (np.array(embeddings), np.array(labels),
            alpha_spat, alpha_morph, alpha_stream)


def fit_svm(X_train, y_train):
    """
    Fit SVM with RBF kernel. Manual inner 5-fold CV — no joblib, no
    multiprocessing, no CUDA fork segfault. Evaluates all C x gamma
    combinations sequentially and returns the best fitted pipeline.
    """
    from sklearn.model_selection import StratifiedKFold as _SKF
    from sklearn.preprocessing import StandardScaler as _SS
    from sklearn.svm import SVC as _SVC
    from sklearn.metrics import roc_auc_score as _auc
    import numpy as _np

    C_values     = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]  # finer grid around C=1 which consistently wins
    gamma_values = ["scale", "auto"]  # RBF kernel only

    cv_splitter = _SKF(n_splits=5, shuffle=True, random_state=GLOBAL_SEED)
    best_score, best_C, best_gamma = -1.0, 1.0, "scale"

    for C in C_values:
        for gamma in gamma_values:
            fold_scores = []
            for tr_idx, va_idx in cv_splitter.split(X_train, y_train):
                Xtr, Xva = X_train[tr_idx], X_train[va_idx]
                ytr, yva = y_train[tr_idx], y_train[va_idx]
                scaler   = _SS(); Xtr = scaler.fit_transform(Xtr); Xva = scaler.transform(Xva)
                clf      = _SVC(kernel="rbf", C=C, gamma=gamma, probability=True,
                                class_weight="balanced", random_state=GLOBAL_SEED)
                clf.fit(Xtr, ytr)
                proba    = clf.predict_proba(Xva)[:, 1]
                try:    fold_scores.append(_auc(yva, proba))
                except: fold_scores.append(0.5)
            mean_score = _np.mean(fold_scores)
            if mean_score > best_score:
                best_score, best_C, best_gamma = mean_score, C, gamma

    # Refit on full training set with best hyperparams
    scaler = _SS()
    X_scaled = scaler.fit_transform(X_train)
    clf = _SVC(kernel="rbf", C=best_C, gamma=best_gamma, probability=True,
               class_weight="balanced", random_state=GLOBAL_SEED)
    clf.fit(X_scaled, y_train)

    print(f"    SVM best: C={best_C}  gamma={best_gamma}  "
          f"inner_AUC={best_score:.4f}")

    # Return a fitted pipeline so predict_proba works on raw (unscaled) test data
    pipe = Pipeline([("scaler", _SS()), ("svc", _SVC(
        kernel="rbf", C=best_C, gamma=best_gamma, probability=True,
        class_weight="balanced", random_state=GLOBAL_SEED))])
    pipe.fit(X_train, y_train)
    return pipe

## 7. Youden's J threshold

In [8]:
def youden_threshold(labels, probs):
    labels = list(labels); probs = list(probs)
    if len(set(labels)) < 2: return 0.5, 0.0, 0.0, 1.0
    try:
        fpr, tpr, thr = roc_curve(labels, probs)
        j   = tpr - fpr
        idx = min(int(np.argmax(j)), len(thr) - 1)
        t   = float(thr[idx])
        if not np.isfinite(t): t = 0.5
        return t, float(j[idx]), float(tpr[idx]), float(1 - fpr[idx])
    except Exception:
        return 0.5, 0.0, 0.5, 0.5

## 8. Single Monte Carlo split: train GNN → fit SVM → evaluate

In [9]:
def run_split(train_list, test_list, device, model_kwargs, split_idx):
    print(f"\n  -- Split {split_idx+1} --")
    n_pmci_tr = sum(1 for d in train_list if int(d.y.item()) == 1)
    n_pmci_te = sum(1 for d in test_list  if int(d.y.item()) == 1)
    print(f"    Train={len(train_list)} (pMCI={n_pmci_tr})  "
          f"Test={len(test_list)} (pMCI={n_pmci_te})")

    # ── Oversample minority in training ──────────────────────────────────
    train_aug = oversample_minority(train_list, minority_label=1)

    # ── Build and train GNN ───────────────────────────────────────────────
    seed = GLOBAL_SEED + split_idx * 13
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)

    # [MOD 2] Auto-detect global_feat_dim from dataset
    sample = train_list[0]
    global_feat_dim = int(sample.x_global.shape[0]) if hasattr(sample, "x_global") else None
    kwargs_with_global = dict(model_kwargs, global_feat_dim=global_feat_dim)

    model = BrainGATv2(**kwargs_with_global).to(device)
    aug_labels = torch.tensor([d.y.item() for d in train_aug])
    criterion  = build_focal(aug_labels, device)
    opt, sched = build_optimizer(model)

    train_gnn(model, train_aug, criterion, opt, sched, device)

    # ── Extract embeddings ────────────────────────────────────────────────
    X_tr, y_tr, _, _, _ = extract_embeddings(model, train_list, device)
    X_te, y_te, a_spat_te, a_morph_te, a_stream_te = \
        extract_embeddings(model, test_list, device)

    # ── Fit SVM on train embeddings ───────────────────────────────────────
    svm = fit_svm(X_tr, y_tr)

    # ── Predict on test ───────────────────────────────────────────────────
    probs = svm.predict_proba(X_te)[:, 1]
    thr, j, sens, spec = youden_threshold(y_te.tolist(), probs.tolist())
    preds = (probs >= thr).astype(int)

    try:
        auc = roc_auc_score(y_te, probs)
        fpr_arr, tpr_arr, _ = roc_curve(y_te, probs)
    except Exception:
        auc = float("nan")
        fpr_arr = tpr_arr = np.array([0.0, 1.0])

    try:
        cm = confusion_matrix(y_te, preds, labels=[0, 1])
    except Exception:
        cm = np.zeros((2, 2), dtype=int)

    result = {
        "auc":          auc,
        "bal_acc":      (sens + spec) / 2.0,
        "sensitivity":  sens,
        "specificity":  spec,
        "youden_j":     j,
        "f1_pmci":      f1_score(y_te, preds, pos_label=1, zero_division=0),
        "f1_smci":      f1_score(y_te, preds, pos_label=0, zero_division=0),
        "cm":           cm,
        "threshold":    thr,
        "probs":        probs.tolist(),
        "labels":       y_te.tolist(),
        "fpr":          fpr_arr,
        "tpr":          tpr_arr,
        # Full attention payload for LLM explainability
        "alpha_spat":   a_spat_te,
        "alpha_morph":  a_morph_te,
        "alpha_stream": a_stream_te,
        "split":        split_idx + 1,
    }

    print(f"    AUC={auc:.4f}  BalAcc={result['bal_acc']:.4f}  "
          f"Sens={sens:.3f}  Spec={spec:.3f}  F1-pMCI={result['f1_pmci']:.4f}")
    print(f"    CM: TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}")

    return result

## 9. Monte Carlo Cross-Validation

In [10]:
N_SPLITS = 10

def run_mccv(full_dataset, device, model_kwargs):
    all_labels = [full_dataset[i].y.item() for i in range(len(full_dataset))]
    all_data   = [full_dataset[i] for i in range(len(full_dataset))]

    n_pmci = int(sum(all_labels))
    n_smci = int(len(all_labels) - n_pmci)
    print(f"\nDataset: {len(full_dataset)} patients  pMCI={n_pmci}  sMCI={n_smci}")

    if len(model_kwargs):
        tmp = BrainGATv2(**model_kwargs)
        # Exclude aux_mlp from count for reporting (SVM replaces it)
        gnn_params = sum(p.numel() for n, p in tmp.named_parameters()
                         if "aux_mlp" not in n)
        total_params = sum(p.numel() for p in tmp.parameters())
        print(f"GNN params (backbone): {gnn_params:,}  "
              f"(+aux MLP {total_params-gnn_params:,} discarded at inference)")
        del tmp

    sss = StratifiedShuffleSplit(
        n_splits=N_SPLITS, test_size=0.2, random_state=GLOBAL_SEED)

    all_results = []
    for si, (tr_idx, te_idx) in enumerate(
            sss.split(range(len(all_data)), all_labels)):
        train_list = [all_data[i] for i in tr_idx]
        test_list  = [all_data[i] for i in te_idx]
        result = run_split(train_list, test_list, device, model_kwargs, si)
        all_results.append(result)

    return all_results

## 10. Summary and plots

In [11]:
def print_summary(results):
    metrics = ["auc", "bal_acc", "sensitivity", "specificity",
               "f1_pmci", "f1_smci", "youden_j"]
    labels  = ["AUC-ROC    ", "Bal. Acc   ", "Sensitivity",
               "Specificity", "F1  pMCI  ", "F1  sMCI  ", "Youden's J "]

    print(f"\n{'='*60}")
    print(f"MONTE CARLO CV SUMMARY  ({N_SPLITS} splits, mean ± std)")
    print(f"{'='*60}")
    for m, lbl in zip(metrics, labels):
        vals = [r[m] for r in results if not math.isnan(r[m])]
        print(f"  {lbl}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

    total_cm = sum(r["cm"] for r in results)
    all_labels_c, all_preds_c = [], []
    for r in results:
        preds_f = [1 if p >= r["threshold"] else 0 for p in r["probs"]]
        all_labels_c.extend(r["labels"])
        all_preds_c.extend(preds_f)

    print(f"\n  Aggregated Confusion Matrix (all {N_SPLITS} splits):")
    print(f"               Pred sMCI   Pred pMCI")
    print(f"  True sMCI :  {total_cm[0,0]:>8d}  {total_cm[0,1]:>9d}")
    print(f"  True pMCI :  {total_cm[1,0]:>8d}  {total_cm[1,1]:>9d}")
    print(f"\n  Combined Classification Report:")
    print(classification_report(all_labels_c, all_preds_c,
                                target_names=["sMCI", "pMCI"], zero_division=0))
    print("=" * 60)


def plot_all(results, prefix="neurograph_v5"):
    BG, AX = "#0f0f0f", "#1a1a2e"
    CLR = plt.cm.tab10(np.linspace(0, 1, N_SPLITS))

    # ── ROC curve with mean ± std band ────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=BG)
    ax.set_facecolor(AX)
    mean_fpr = np.linspace(0, 1, 200)
    tprs = []
    for i, r in enumerate(results):
        fpr = np.array(r["fpr"]); tpr = np.array(r["tpr"])
        ax.plot(fpr, tpr, alpha=0.35, lw=1, color=CLR[i],
                label=f"Split {i+1} (AUC={r['auc']:.3f})")
        interp = np.interp(mean_fpr, fpr, tpr)
        interp[0] = 0.0
        tprs.append(interp)
    mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
    std_tpr  = np.std(tprs,  axis=0)
    auc_vals = [r["auc"] for r in results if not math.isnan(r["auc"])]
    ax.plot(mean_fpr, mean_tpr, "white", lw=2.5,
            label=f"Mean AUC={np.mean(auc_vals):.3f}±{np.std(auc_vals):.3f}")
    ax.fill_between(mean_fpr, mean_tpr - std_tpr, mean_tpr + std_tpr,
                    color="white", alpha=0.10, label="±1 std")
    ax.plot([0, 1], [0, 1], ls="--", color="#555", lw=1)
    ms = np.mean([r["sensitivity"] for r in results])
    mp = np.mean([r["specificity"] for r in results])
    ax.scatter(1 - mp, ms, s=100, zorder=5, color="#FFD54F",
               edgecolors="white", lw=0.8,
               label=f"Mean Youden J  Sens={ms:.3f} Spec={mp:.3f}")
    ax.set_xlabel("FPR", color="#ccc", fontsize=11)
    ax.set_ylabel("TPR", color="#ccc", fontsize=11)
    ax.set_title("NeuroGraph-X v5 — ROC (10-split MCCV)\npMCI vs sMCI  |  GNN + SVM",
                 color="white", fontsize=12, fontweight="bold")
    ax.tick_params(colors="#aaa")
    for sp in ax.spines.values(): sp.set_edgecolor("#444")
    ax.legend(facecolor="#111", labelcolor="white", fontsize=7,
              framealpha=0.9, loc="lower right", ncol=2)
    ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.01)
    plt.tight_layout()
    plt.savefig(f"{prefix}_roc.png", dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(); print(f"  Saved: {prefix}_roc.png")

    # ── Per-split metric bar chart ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), facecolor=BG)
    split_lbls = [f"S{r['split']}" for r in results]
    for ai, (metric, ylabel) in enumerate([
            ("auc",     "AUC-ROC"),
            ("f1_pmci", "F1 pMCI"),
            ("bal_acc", "Balanced Acc")]):
        ax = axes[ai]; ax.set_facecolor(AX)
        vals = [r[metric] for r in results]
        bars = ax.bar(split_lbls, vals, color=CLR, width=0.6)
        ax.axhline(np.mean(vals), color="white", lw=1.5, ls="--",
                   label=f"Mean={np.mean(vals):.3f}")
        ax.set_ylim(0, 1.05)
        ax.set_title(ylabel, color="white", fontsize=11)
        ax.tick_params(colors="#aaa", labelsize=8)
        ax.set_ylabel(ylabel, color="#aaa", fontsize=9)
        for sp in ax.spines.values(): sp.set_edgecolor("#444")
        ax.legend(facecolor="#111", labelcolor="white", fontsize=8)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01,
                    f"{v:.2f}", ha="center", va="bottom",
                    color="white", fontsize=7)
    fig.suptitle("NeuroGraph-X v5 — Per-Split Metrics (GNN + SVM)",
                 color="white", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{prefix}_per_split.png", dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(); print(f"  Saved: {prefix}_per_split.png")

    # ── Aggregated confusion matrix ────────────────────────────────────────
    total_cm = sum(r["cm"] for r in results).astype(float)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), facecolor=BG)
    for ai, (cm_d, title, fmt) in enumerate([
            (total_cm,
             "Aggregated Counts", "{:.0f}"),
            (total_cm / total_cm.sum(axis=1, keepdims=True),
             "Row-Normalised",    "{:.2f}")]):
        ax = axes[ai]; ax.set_facecolor(AX)
        im = ax.imshow(cm_d, cmap="Blues", vmin=0, vmax=cm_d.max())
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cb.ax.tick_params(colors="#aaa", labelsize=8)
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(["sMCI", "pMCI"], color="white", fontsize=11)
        ax.set_yticklabels(["sMCI", "pMCI"], color="white", fontsize=11)
        ax.set_xlabel("Predicted", color="#ccc", fontsize=11)
        ax.set_ylabel("Actual",    color="#ccc", fontsize=11)
        ax.set_title(title, color="white", fontsize=11, fontweight="bold")
        for sp in ax.spines.values(): sp.set_edgecolor("#444")
        for row in range(2):
            for col in range(2):
                v = cm_d[row, col]
                ax.text(col, row, fmt.format(v), ha="center", va="center",
                        color="white" if v < cm_d.max() * 0.6 else "#111",
                        fontsize=13, fontweight="bold")
    fig.suptitle(
        f"NeuroGraph-X v5 — Confusion Matrix ({N_SPLITS} splits, "
        f"N={int(total_cm.sum())})",
        color="white", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{prefix}_cm.png", dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(); print(f"  Saved: {prefix}_cm.png")

    # ── AUC per split bar chart ────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 4), facecolor=BG)
    ax.set_facecolor(AX)
    auc_vals = [r["auc"] for r in results]
    ax.bar(range(1, N_SPLITS + 1), auc_vals,
           color=[CLR[i] for i in range(N_SPLITS)], width=0.6)
    ax.axhline(np.mean(auc_vals), color="white", lw=2, ls="--",
               label=f"Mean AUC={np.mean(auc_vals):.3f}")
    ax.axhline(0.5,  color="#888",    lw=1, ls=":",   label="Chance")
    ax.axhline(0.75, color="#FFD54F", lw=1, ls="-.",  label="Target AUC=0.75")
    ax.set_ylim(0, 1.05)
    ax.set_xticks(range(1, N_SPLITS + 1))
    ax.set_xlabel("Split",    color="#ccc")
    ax.set_ylabel("AUC-ROC",  color="#ccc")
    ax.set_title("AUC per Split — NeuroGraph-X v5",
                 color="white", fontweight="bold")
    ax.tick_params(colors="#aaa")
    for sp in ax.spines.values(): sp.set_edgecolor("#444")
    ax.legend(facecolor="#111", labelcolor="white", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{prefix}_auc_dist.png", dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(); print(f"  Saved: {prefix}_auc_dist.png")

## 11. Run

In [12]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    full_dataset = NeuroGraphDataset(
        root='./data',
        csv_path='./data/GNN_Target_Labels.csv',
        spatial_matrix_path='./data/A_spatial_master.pt',
    )

    MODEL_KWARGS = dict(
        in_channels     = 512,
        proj_dim        = 64,
        hidden_channels = 16,    # stream_dim = 64
        heads           = 4,
        edge_dim_spat   = 1,
        edge_dim_morph  = 1,
        dropout_proj    = 0.2,
        dropout_gat     = 0.3,
    )

    results = run_mccv(full_dataset, device, MODEL_KWARGS)

    print_summary(results)

    print("\nGenerating plots...")
    plot_all(results, prefix="neurograph_v5")
    print("Done.")

Device: cuda

Dataset: 272 patients  pMCI=50  sMCI=222
GNN params (backbone): 105,027  (+aux MLP 2,113 discarded at inference)

  -- Split 1 --
    Train=217 (pMCI=40)  Test=55 (pMCI=10)
  Oversampling: 40 pMCI×4 → pMCI=160  sMCI=177  total=337
    Epoch   1/150  loss=0.0924  lr=2.10e-05
    Epoch  10/150  loss=0.0875  lr=1.00e-04
    Epoch  20/150  loss=0.0888  lr=1.98e-04
    Epoch  30/150  loss=0.0878  lr=1.98e-04
    Epoch  40/150  loss=0.0870  lr=1.90e-04
    Epoch  50/150  loss=0.0865  lr=1.76e-04
    Epoch  60/150  loss=0.0871  lr=1.58e-04
    Epoch  70/150  loss=0.0866  lr=1.37e-04
    Epoch  80/150  loss=0.0859  lr=1.13e-04
    Epoch  90/150  loss=0.0861  lr=8.84e-05
    Epoch 100/150  loss=0.0865  lr=6.46e-05
    Epoch 110/150  loss=0.0855  lr=4.29e-05
    Epoch 120/150  loss=0.0857  lr=2.46e-05
    Epoch 130/150  loss=0.0853  lr=1.09e-05
    Epoch 140/150  loss=0.0854  lr=2.65e-06
    Epoch 150/150  loss=0.0855  lr=2.30e-07
    SVM best: C=0.1  gamma=scale  inner_AUC=0.6480


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score

# 1. Flatten the raw ResNet features for every patient
X_raw = []
y_raw = []
for p in full_dataset:
    # Average the 166 regional vectors into a single 512-dim vector for the patient
    patient_feature = p.x.mean(dim=0).numpy() 
    X_raw.append(patient_feature)
    y_raw.append(p.y.item())

X_raw = np.array(X_raw)
y_raw = np.array(y_raw)

# 2. Run a simple, robust baseline
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)

# 3. Evaluate
scores = cross_val_score(rf, X_raw, y_raw, cv=5, scoring="roc_auc")
print(f"Raw Feature Baseline AUC: {scores.mean():.4f} ± {scores.std():.4f}")

Raw Feature Baseline AUC: 0.5385 ± 0.0745
